# Using Predictive Analytics to Identify Engagement Drivers in Instagram Reels
## Notebook 2 of 4: Data Preparation
**CRISP-DM phase 3**  |  Previous: `01_business_and_data_understanding`  |  Next: `03_modeling`

**Input:** `../Data/raw_data.csv`  |  **Output:** `../Data/cleaned_data.csv`

The CRISP-DM data preparation tasks are carried out in order: **select** the data, **clean** it, **construct** new features, and **format** it for modeling.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

reels_df = pd.read_csv('../Data/raw_data.csv')
print(f"Raw data: {reels_df.shape[0]} rows, {reels_df.shape[1]} columns")

## 3.1 Select data

### Roles of the variables
| Role | Variables | Used as predictors? |
|---|---|---|
| **Target** | `performance_bucket` | n/a |
| **Content framing and structure** | `hook_category`, `reel_structure`, `craft_category` | Yes |
| **Delivery mechanics** | `result_shown_first`, `voice_present`, `audio_not_required`, `pov_present` | Yes |
| **Context** (kept in the dataset, not used as predictors) | `creator_handle`, `total_followers`, `date_posted`, `collection_date`, `days_since_posted` | No. The target is already creator-relative, which accounts for audience size, and Reel age is treated as a confounder (see Notebook 1, section 2.4) |
| **Outcome measures** (excluded to avoid leakage) | `views`, `likes`, `comments`, `engagement_rate`, `like_rate`, `comment_rate`, `views_per_follower` | No. They are components of the engagement outcome, so using them to predict it would be circular |
| **Identifier** | `reel_id` (extracted from `url`) | No |
| **Dropped** | `url`, `hook_text` | No. `url` is only needed to extract the Reel ID, and `hook_text` is free text that is missing when the hook is visual |

### Create a unique Reel ID

In [ ]:
# The Reel ID is the segment after /reel/, /reels/, or /p/ in the URL,
# e.g. https://www.instagram.com/reel/DT5we1ZAQS4/
reels_df['reel_id'] = reels_df['url'].str.extract(r'instagram\.com/(?:reel|reels|p)/([^/?#]+)', expand=False)

# A URL with no Reel ID (for example a profile link) keeps its full URL as the identifier
no_id = reels_df['reel_id'].isna()
print("URLs without a Reel ID (full URL kept as the identifier):", no_id.sum())
reels_df.loc[no_id, 'reel_id'] = reels_df.loc[no_id, 'url']

### Resolve duplicate Reels and remove the placeholder hook
Notebook 1 found Reels that appear twice with identical content but **opposite performance labels**. A Reel cannot be both baseline and high-performing, and the copies would also land in both the training and validation sets, so they must be resolved before modeling. Reels that are exact duplicates (same label) keep a single copy.

Which label is correct depends on how the data was collected, so the rule is a setting:
- `'drop_all'` (default) removes every copy of a conflicting Reel, because the label cannot be trusted.
- `'keep_high'` keeps the copy labeled high-performing (for example, if baseline Reels were sampled from recent posts that may also be top posts).

One Reel also has the unfilled placeholder `Choose from...` as its hook category, so it is removed.

In [ ]:
DUPLICATE_RULE = 'drop_all'   # or 'keep_high'

rows_before = len(reels_df)

# 1) exact duplicates (same URL and same label): keep one copy
reels_df = reels_df.drop_duplicates(subset=['url', 'performance_bucket'])

# 2) URLs that still appear more than once have conflicting labels
conflict = reels_df['url'].duplicated(keep=False)
print(f"Reels with conflicting labels: {reels_df.loc[conflict, 'url'].nunique()} ({conflict.sum()} rows)")

if DUPLICATE_RULE == 'drop_all':
    reels_df = reels_df[~conflict]
elif DUPLICATE_RULE == 'keep_high':
    is_high = reels_df['performance_bucket'].astype(str).str.strip().str.lower().eq('high')
    reels_df = reels_df.assign(_is_high=is_high).sort_values('_is_high', ascending=False)
    reels_df = reels_df.drop_duplicates(subset='url').sort_index().drop(columns='_is_high')
else:
    raise ValueError("DUPLICATE_RULE must be 'drop_all' or 'keep_high'")

# 3) remove the unfilled dropdown placeholder in hook_category
placeholder = reels_df['hook_category'].str.contains('choose from', case=False, na=False)
print(f"Placeholder hook rows removed: {placeholder.sum()}")
reels_df = reels_df[~placeholder]

print(f"Rows: {rows_before} -> {len(reels_df)}")
assert not reels_df['url'].duplicated().any()

# drop columns that are not features, and put the identifier first
reels_df = reels_df.drop(columns=['url', 'hook_text'])
reels_df = reels_df[['reel_id'] + [c for c in reels_df.columns if c != 'reel_id']].reset_index(drop=True)

## 3.2 Clean data

### Rename columns

In [ ]:
reels_df = reels_df.rename(columns={
    'views_raw': 'views',
    'likes_raw': 'likes',
    'comments_raw': 'comments'
})

### Standardize text and encode Y/N values
Category labels are converted to lowercase, trimmed, and joined with underscores (for example `Fine Art (Paint)` becomes `fine_art_paint`). The four delivery variables are encoded as 1 (`Y`) and 0 (`N`).

In [ ]:
categorical_cols = ['craft_category', 'reel_structure', 'hook_category']
binary_cols = ['pov_present', 'result_shown_first', 'voice_present', 'audio_not_required']

for col in categorical_cols:
    reels_df[col] = (
        reels_df[col].astype(str).str.lower().str.strip()
        .str.replace(' / ', '_', regex=False)
        .str.replace(' ', '_', regex=False)
        .str.replace('[()]', '', regex=True)
    )

for col in binary_cols:
    reels_df[col] = (
        reels_df[col].astype(str).str.strip().str.upper().map({'Y': 1, 'N': 0})
    )

assert reels_df[binary_cols].notna().all().all(), "Unexpected value in a Y/N column"

### Convert counts to numbers
`total_followers`, `views`, `likes`, and `comments` are stored as text such as `953K` or `1.2M`.

In [ ]:
def convert_to_number(x):
    """Convert strings such as '113K' or '1.2M' to numbers."""
    x = str(x).strip().upper().replace(',', '')
    if 'K' in x:
        return float(x.replace('K', '')) * 1_000
    if 'M' in x:
        return float(x.replace('M', '')) * 1_000_000
    return float(x)

for col in ['total_followers', 'views', 'likes', 'comments']:
    reels_df[col] = reels_df[col].apply(convert_to_number)

### Standardize dates and the target
`date_posted` appears in more than one format, so it is parsed with `format='mixed'`. The check afterwards confirms that every date was parsed. The target is encoded as 0 (baseline) and 1 (high-performing).

In [ ]:
reels_df['date_posted'] = pd.to_datetime(
    reels_df['date_posted'].astype(str).str.strip(), format='mixed', errors='coerce'
)
reels_df['collection_date'] = pd.to_datetime(
    reels_df['collection_date'].astype(str).str.strip(), format='%m/%d/%Y'
)
assert reels_df['date_posted'].notna().all(), "Some date_posted values could not be parsed"

reels_df['performance_bucket'] = (
    reels_df['performance_bucket'].astype(str).str.strip().str.lower().map({'baseline': 0, 'high': 1})
)
assert reels_df['performance_bucket'].notna().all(), "Unexpected value in performance_bucket"

Beyond the duplicate and placeholder rows removed above, no other records were removed. The remaining missing values were expected (see Notebook 1), and no outliers were removed because extreme engagement values are meaningful in the context of social media performance.

## 3.3 Construct data

### Derived features
`engagement_rate` is recalculated from likes, comments, and views, because the raw column is almost entirely empty. The age of the Reel at collection is calculated in days (weeks would carry exactly the same information).

In [ ]:
# How old the Reel was when performance was measured
reels_df['days_since_posted'] = (reels_df['collection_date'] - reels_df['date_posted']).dt.days

# Outcome measures: normalized so reels from large and small creators can be compared
reels_df['engagement_rate'] = (reels_df['likes'] + reels_df['comments']) / reels_df['views']
reels_df['like_rate'] = reels_df['likes'] / reels_df['views']
reels_df['comment_rate'] = reels_df['comments'] / reels_df['views']
reels_df['views_per_follower'] = reels_df['views'] / reels_df['total_followers']

### Interaction features
Notebook 1 (section 2.4f) showed that some category pairs behave differently from their single-feature rates. Each interaction feature below equals 1 when a Reel has **both** values in the pair, and 0 otherwise. They are used in the second modeling iteration in Notebook 3.

In [ ]:
# (column A, value A, column B, value B)
# NOTE: reconstructed from the interaction columns used in the evaluation step; verify against
# the original interaction analysis and add or remove pairs as needed.
INTERACTIONS = [
    ('craft_category', 'home_decor',       'hook_category',  'direct_value_delivery'),
    ('craft_category', 'holiday_seasonal', 'reel_structure', 'process'),
    ('craft_category', 'upcycling',        'reel_structure', 'process'),
    ('craft_category', 'plants_garden',    'reel_structure', 'process'),
    ('craft_category', 'home_decor',       'reel_structure', 'showcase'),
    ('hook_category',  'curiosity_gap',        'reel_structure', 'process'),
    ('hook_category',  'direct_value_delivery', 'reel_structure', 'showcase'),
    ('hook_category',  'transformation_tease',  'reel_structure', 'process'),
]

for col_a, val_a, col_b, val_b in INTERACTIONS:
    reels_df[f"int_{val_a}_{val_b}"] = ((reels_df[col_a] == val_a) & (reels_df[col_b] == val_b)).astype(int)

interaction_cols = [c for c in reels_df.columns if c.startswith('int_')]
print("Reels with each interaction:")
print(reels_df[interaction_cols].sum().sort_values(ascending=False))

## 3.4 Format data

Each categorical variable is converted into indicator (0/1) columns. All categories are kept, so every indicator can be read directly (for example `hook_category_curiosity_gap = 1` means the hook is a curiosity gap). Regularization in the logistic regression handles the redundancy between indicator columns.

In [ ]:
reels_df = pd.get_dummies(
    reels_df,
    columns=categorical_cols,
    drop_first=False,
    dtype=int
)

reels_df.head()

### Dimension reduction considerations
- **Correlated numeric variables.** Views, likes, and comments are strongly correlated (Notebook 1, section 2.4c). They are outcome measures and are not predictors, so no further variables were removed.
- **PCA.** Principal Component Analysis was considered but not applied. PCA is most useful for reducing a large number of correlated continuous variables, whereas this dataset consists mainly of categorical features converted to binary indicators plus a few numeric variables. PCA would also reduce interpretability, which matters here because the goal is to understand how specific content features relate to performance.

## 3.5 Validate and save the prepared dataset

In [ ]:
print("Shape:", reels_df.shape)
print("Missing values in the prepared data:", int(reels_df.isna().sum().sum()))
print("\nTarget distribution:")
print(reels_df['performance_bucket'].value_counts(normalize=True).round(3))

print("\nColumn groups:")
print("  Binary features:   ", binary_cols)
print("  Category indicators:", len([c for c in reels_df.columns if c.startswith(('craft_category_', 'reel_structure_', 'hook_category_'))]))
print("  Interaction columns:", len(interaction_cols))

In [ ]:
# Export the prepared dataset (read by Notebook 3)
reels_df.to_csv('../Data/cleaned_data.csv', index=False)

### Summary of preparation steps
1. Extracted a unique `reel_id` from the URL, resolved Reels that appeared twice with conflicting labels, removed the placeholder hook row, and dropped `url` and `hook_text`.
2. Renamed columns, standardized category labels, and encoded the `Y`/`N` variables as 1/0.
3. Converted `K`/`M` text counts to numbers and parsed the dates.
4. Encoded the target as 0 (baseline) and 1 (high-performing).
5. Calculated Reel age, engagement rate, like rate, comment rate, and views per follower.
6. Built the interaction features and one-hot encoded the categorical variables.
7. Validated the result (no missing values) and saved it for modeling.